# Bluestock Fintech - Mutual Fund Analytics Capstone
## Day 1: Project Setup, Data Ingestion & AMFI Validation
**Author:** Bluestock Analytics Team  
**Objective:** Ingest 10 AMFI mutual fund CSV datasets, profile shapes/dtypes/anomalies, extract live NAV from `mfapi.in` REST API, and validate AMFI code referential integrity.


In [ ]:
import os
import sys
import requests
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = Path('..').resolve() if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

print(f'Working directory: {BASE_DIR}')


### Task 1: Verify Directory Structure
Ensure required directories (`data/raw`, `data/processed`, `notebooks`, `sql`, `dashboard`, `reports`) are properly initialized.


In [ ]:
required_dirs = ['data/raw', 'data/processed', 'notebooks', 'sql', 'dashboard', 'reports']
for d in required_dirs:
    p = BASE_DIR / d
    p.mkdir(parents=True, exist_ok=True)
    print(f'Verified directory: {d}')


### Task 2: Verify Installed Dependencies & requirements.txt


In [ ]:
packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly', 'sqlalchemy', 'requests', 'scipy', 'jupyter']
for pkg in packages:
    mod = __import__(pkg)
    print(f"{pkg:<15}: {getattr(mod, '__version__', 'installed')}")


### Task 3: Load All 10 CSV Datasets & Profile Shapes, Dtypes & Anomalies


In [ ]:
csv_files = [
    '01_fund_master.csv',
    '02_nav_history.csv',
    '03_aum_by_fund_house.csv',
    '04_monthly_sip_inflows.csv',
    '05_category_inflows.csv',
    '06_industry_folio_count.csv',
    '07_scheme_performance.csv',
    '08_investor_transactions.csv',
    '09_portfolio_holdings.csv',
    '10_benchmark_indices.csv'
]

dfs = {}
for fname in csv_files:
    fpath = BASE_DIR / fname if (BASE_DIR / fname).exists() else RAW_DIR / fname
    df = pd.read_csv(fpath)
    dfs[fname] = df
    print(f'=== {fname} ===')
    print(f'Shape: {df.shape}')
    print('Dtypes:')
    for col, dt in df.dtypes.items():
        print(f'  {col}: {dt}')
    nulls = df.isnull().sum()[df.isnull().sum() > 0]
    print(f'Missing Values: {dict(nulls) if not nulls.empty else "None"}')
    print(f'Duplicates: {df.duplicated().sum()}')
    print('Head (2 rows):')
    display(df.head(2))
    print('-' * 60)


### Tasks 4 & 5: Fetch Live NAV from mfapi.in API
Fetch anchor scheme (HDFC Top 100 - AMFI 125497) and 5 key schemes (SBI 119551, ICICI 120503, Nippon 118632, Axis 119092, Kotak 120841).


In [ ]:
import sys
sys.path.append(str(BASE_DIR))
import live_nav_fetch
summary_df = live_nav_fetch.run_live_nav_pipeline()
display(summary_df)


### Task 6: Explore Fund Master & AMFI Scheme Code Architecture


In [ ]:
fm = dfs['01_fund_master.csv']

print(f'Total Schemes: {len(fm)}')
print(f'Fund Houses ({fm["fund_house"].nunique()}):')
print(fm['fund_house'].value_counts())

print(f'\nCategories ({fm["category"].nunique()}):')
print(fm['category'].value_counts())

print(f'\nSub-Categories ({fm["sub_category"].nunique()}):')
print(fm['sub_category'].value_counts())

print(f'\nRisk Grades ({fm["risk_category"].nunique()}):')
print(fm['risk_category'].value_counts())

print(f'\nExpense Ratio Range: {fm["expense_ratio_pct"].min()}% - {fm["expense_ratio_pct"].max()}% (Avg: {fm["expense_ratio_pct"].mean():.2f}%)')


### Task 7: Validate AMFI Scheme Codes
Confirm every code in `01_fund_master.csv` exists in `02_nav_history.csv`.


In [ ]:
fm_codes = set(dfs['01_fund_master.csv']['amfi_code'])
nh_codes = set(dfs['02_nav_history.csv']['amfi_code'])

in_master_not_nav = fm_codes - nh_codes
in_nav_not_master = nh_codes - fm_codes

print(f'Unique codes in fund_master : {len(fm_codes)}')
print(f'Unique codes in nav_history : {len(nh_codes)}')
print(f'Missing in NAV: {in_master_not_nav}')
print(f'Extra in NAV  : {in_nav_not_master}')

assert len(in_master_not_nav) == 0 and len(in_nav_not_master) == 0, 'AMFI codes mismatch!'
print('PASS: 100% AMFI Scheme Code Referential Integrity Verified!')
